In [ ]:
# Для установки библиотеки воспользуйтесь командой
# !conda install conda-forge::transformers
# или
# !pip install transformers

In [ ]:
# Любая LLM работает с токенами, поэтому нам нужна не только модель, но и токенизатор
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Зададим название модели из репозитория huggingface
# список доступных моделей можно посмотреть по ссылке https://huggingface.co/models
# будем использовать модель Qwen или YandexGPT

model_name = "Qwen/Qwen2.5-7B-Instruct-1M"
#model_name = "yandex/YandexGPT-5-Lite-8B-instruct"

# В названии модели обычно указываются - версия (2,5) количество параметров (7 млрд.)
# этап обучения или предназначение (Base, Instruct, Code, Thinking и т.п.)
# размер контекстного окна (1 млн.)
# также могут указываться параметры квантования (FP8, GGUF, ) и другие характеристики.


# Загружаем предобученную можель
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# Загружаем предобученный токенизатор
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Загружаем текстовый файл преобразуя его в строку
file = 'Article.txt'
with open(file, 'r', encoding='cp1251') as file:
    data = file.read().replace('\n', ' ')

print(len(data))

20128


In [ ]:
# Посмотрим выдержку из файла
print(data[90:160])

 История развития современных нейросетей: хронология, ключевые модели 


In [ ]:
# Посчитаем количество слов в файле чтобы примерно сопоставить с контекстным окном
num_words = len(data.split())
print(num_words)

2677


In [ ]:
# Для общения с LLM нужно составить промпт
# он будет состоять из системной части - наши инструкции что нужно сделать
# и пользовательской части - текста файла
messages = [
    {"role": "system", "content": "Суммаризируй текст. Определи жанр текста, выдели основную информацию. Ответ давай на русском языке."},
    {"role": "user", "content": data[0:5000]} # берем только часть чтобы долго не ждать генерацию
]

# Для YandexGPT инструкции пишутся от имени пользователя
#messages = [
#    {"role": "user", "content": "Суммаризируй текст. Определи жанр текста, выдели основную информацию. Ответ давай на русском языке."+data[0:5000]} # берем только часть чтобы долго не ждать генерацию
#]

# Метод apply_chat_template() используется для форматирования сообщений в одну строку, формат
# которой ориентирован на использование чат-ориентированных языковых моделей.
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# В текст добавляются специальные  токены-метки для указания структуры разговора.
print(text[0:160])

<|im_start|>system
Суммаризируй текст. Определи жанр текста, выдели основную информацию. Ответ давай на русском языке.<|im_end|>
<|im_start|>user
Статья доступн


In [ ]:
# Токенизатор разбивает текст на токены.
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# input_ids — токены в числовом виде, которые представляют собой уникальный номер (ID) из словаря модели.
# attention_mask — задает маску, которая показывает позиции реальных токенов и дополнений (padding) или служебных токенов.
# Модели обрабатывают батчи текстов одинаковой длины. Чтобы выровнять длину реальных текстов короткие тексты дополняются специальным токеном [PAD].
# Таким токенам в attention_mask соответствуют нули, чтобы модель их не обрабатывала.
print(model_inputs)

{'input_ids': tensor([[151644,   8948,    198,  ..., 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}


In [ ]:
# Токенизированный текст подаем в модель.
# max_new_tokens - задает максимальное число генерируемых в ответе токенов.
# Можно указать и другие параметры:
# temperature - регулирует "случайность" в выборе следующего токена (<1 — более предсказуемо, >1 — более хаотично).
# top_k - ограничивает выбор следующего токена K наиболее вероятными
# repetition_penalty - штраф за повторяющиеся токены (>1 - штраф, 1 - без штрафа).
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=1024,
#    temperature=0.9,
#    top_k=50,
#    repetition_penalty=1.2
)

In [ ]:
# Так как модель возвращает и промпт и сгенерированные токены, выделяем только ответ.
generated_ids_ = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# Преобразуем ID токенов обратно в слова.
response = tokenizer.batch_decode(generated_ids_, skip_special_tokens=True)[0]

In [ ]:
# Смотрим что получилось.
print(response)

### Жанр текста:
Это статья о развитии нейронных сетей, которая сочетает в себе исторический обзор и анализ текущего состояния технологий. Жанр можно охарактеризовать как научно-популярную статью.

### Основная информация:

1. **История развития нейронных сетей**:
   - В середине XX века была предложена математическая модель нейрона.
   - В 1950-х годах был представлен персептрон — первая практическая реализация нейросети.
   - В 1980-2000-х годах начали развиваться алгоритмы обучения, сравнения и анализа данных.
   - В 2000-х годах появилось Deep Learning благодаря мощным графическим процессорам и доступности больших данных.

2. **Развитие современных нейронных сетей на западе**:
   - В 2015 году была основана OpenAI, некоммерческая организация, занимающаяся исследованиями в области ИИ.
   - В 2016 году была выпущена платформа OpenAI Gym для разработки и сравнения алгоритмов обучения с подкреплением.
   - В 2017 году были представлены ИИ-боты для игры в Dota 2.
   - В 2020 году была о